In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm
from joblib import dump

## Functions

In [ ]:
def plot_binary_distribution(
    df,
    value_col,
    label_col="is_slick",
    sentinel_value=-5,
    n_bins=50,
    true_label="True",
    false_label="False",
    true_color="green",
    false_color="red",
    figsize=(8, 5),
    title=None,
):
    # Exclude sentinel / missing values
    mask_valid = df["max_source_collated_score"] != sentinel_value

    vals = df.loc[mask_valid, value_col]
    if vals.empty:
        raise ValueError(f"No valid values found for column '{value_col}'")

    bins = np.linspace(vals.min(), vals.max(), n_bins)

    plt.figure(figsize=figsize)

    plt.hist(
        df.loc[mask_valid & df[label_col], value_col].dropna(),
        bins=bins,
        density=True,
        alpha=0.6,
        label=f"{true_label}",
        color=true_color,
    )

    plt.hist(
        df.loc[mask_valid & ~df[label_col], value_col].dropna(),
        bins=bins,
        density=True,
        alpha=0.6,
        label=f"{false_label}",
        color=false_color,
    )

    plt.xlabel(value_col)
    plt.ylabel("Density")
    plt.title(title or f"Distribution of {value_col} by {label_col}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


# CM


def plot_cm(cm, title="Confusion Matrix"):
    cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots()
    im = ax.imshow(
        cm_normalized, cmap="Blues", alpha=0.6
    )  # <-- alpha sets transparency

    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel("Normalized count", rotation=-90, va="bottom")

    classes = ["Not Slick", "Slick"]
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    for i in range(cm_normalized.shape[0]):
        for j in range(cm_normalized.shape[1]):
            ax.text(
                j,
                i,
                f"{cm[i, j]}\n({cm_normalized[i, j]:.2f})",  # show count + fraction
                ha="center",
                va="center",
                color="black",
                fontsize=12,
            )

    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")
    ax.set_title(title)
    plt.show()


def metrics_vs_threshold(y_prob, y_true, thresholds=None, plot=True, thres=None):
    """
    Compute accuracy, precision, recall, and F1-score across a range of thresholds.

    Parameters:
    -----------
    y_prob : array-like
        Predicted probabilities for the positive class.
    y_true : array-like
        True binary labels (0 or 1).
    thresholds : array-like, optional
        Thresholds to evaluate. Defaults to np.linspace(0,1,101)
    plot : bool, optional
        If True, plots the metrics vs threshold.

    Returns:
    --------
    thresholds : np.ndarray
        Threshold values evaluated.
    metrics_dict : dict
        Dictionary with keys 'accuracy', 'precision', 'recall', 'f1', each containing an array of metric values.
    """
    if thresholds is None:
        thresholds = np.linspace(0, 1, 50)

    accuracies, precisions, recalls, f1s = [], [], [], []

    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        accuracies.append(accuracy_score(y_true, preds))
        precisions.append(precision_score(y_true, preds, zero_division=0))
        recalls.append(recall_score(y_true, preds, zero_division=0))
        f1s.append(f1_score(y_true, preds, zero_division=0))

    metrics_dict = {
        "accuracy": np.array(accuracies),
        "precision": np.array(precisions),
        "recall": np.array(recalls),
        "f1": np.array(f1s),
    }

    if plot:
        plt.figure(figsize=(8, 5))
        plt.plot(thresholds, metrics_dict["accuracy"], marker="o", label="Accuracy")
        plt.plot(thresholds, metrics_dict["precision"], marker="x", label="Precision")
        plt.plot(thresholds, metrics_dict["recall"], marker="s", label="Recall")
        plt.plot(thresholds, metrics_dict["f1"], marker="^", label="F1-score")
        plt.xlabel("Threshold")
        plt.ylabel("Metric value")
        plt.title("Metrics vs Threshold")
        plt.grid(True)
        plt.legend()
        # optional: show current threshold if defined
        try:
            plt.axvline(
                thres, color="gray", linestyle="--", label=f"current thres={thres}"
            )
            plt.legend()
        except NameError:
            pass
        plt.show()

    return thresholds, metrics_dict


import json
from shapely.geometry import shape, LineString, MultiLineString, GeometryCollection
from shapely.ops import unary_union


def geojson_to_lines(geojson_obj):
    if geojson_obj is None:
        return None

    # If stored as string, parse it
    if isinstance(geojson_obj, str):
        geojson_obj = json.loads(geojson_obj)

    if geojson_obj["type"] == "FeatureCollection":
        geoms = [shape(f["geometry"]) for f in geojson_obj["features"]]
        return unary_union(geoms)

    if geojson_obj["type"] == "Feature":
        return shape(geojson_obj["geometry"])

    return shape(geojson_obj)


def plot_lines(geom, ax, **kwargs):
    if geom is None:
        return

    if isinstance(geom, LineString):
        x, y = geom.xy
        ax.plot(x, y, **kwargs)

    elif isinstance(geom, MultiLineString):
        for line in geom.geoms:
            x, y = line.xy
            ax.plot(x, y, **kwargs)

    elif isinstance(geom, GeometryCollection):
        for g in geom.geoms:
            plot_lines(g, ax, **kwargs)

## Load Data

In [ ]:
df_orig = pd.read_csv(
    r"C:\Users\ebeva\SkyTruth\cv3\source potential\initial_testDS.csv"
).drop_duplicates()

In [ ]:
# sample 10% of the slick data
# df_sample = df_orig.sample(frac=0.1, random_state=42)
# df_remain = df_orig.drop(df_sample.index)

In [ ]:
# df_sample.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_baseline_test.csv", index=False)
# df_remain.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_baseline_train.csv", index=False)

In [ ]:
df_test = pd.read_csv(
    r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_baseline_test.csv"
)
df_train = pd.read_csv(
    r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_baseline_train.csv"
)

In [ ]:
# df_feats_added = pd.read_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\hitl_featsAdded_011426.csv").drop_duplicates()
# df_feats_added_sample = df_feats_added[df_feats_added['id'].isin(df_test['id'])]
# df_feats_added_remain = df_feats_added[df_feats_added['id'].isin(df_train['id'])]
# df_feats_added_sample.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_featsAdded_test.csv", index=False)
# df_feats_added_remain.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_featsAdded_train.csv", index=False)

# df_full = pd.read_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\cerulean_detections_all.csv")
# unnattributed_ids = set(df_full['id']) - set(df_orig['id'])
# test_unnattributed_ids_sample = list(unnattributed_ids)[0:int(len(unnattributed_ids)*0.1)]
# train_unnattributed_ids = list(unnattributed_ids)[int(len(unnattributed_ids)*0.1):]
# df_full_sample = df_full[df_full['id'].isin(df_test['id']) | df_full['id'].isin(test_unnattributed_ids_sample)]
# df_full_remain = df_full[df_full['id'].isin(df_train['id']) | df_full['id'].isin(train_unnattributed_ids)]
# df_full_sample.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_full_test.csv", index=False)
# df_full_remain.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_full_train.csv", index=False)

In [ ]:
# df_tst = pd.read_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_full_test.csv")
# df = pd.read_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_full_train.csv")

# data_with_conf = pd.read_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\cerulean_detections_01212025.csv")
# # make train/test splits for data_with_conf using df_tst and df
# data_with_conf_test = data_with_conf[data_with_conf['id'].isin(df_tst['id'])]
# data_with_conf_train = data_with_conf[data_with_conf['id'].isin(df['id'])]

# data_with_conf_test.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_conf_test.csv", index=False)
# data_with_conf_train.to_csv(r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_conf_train.csv", index=False)



In [ ]:
df_full_train = pd.read_csv(
    r"C:\Users\ebeva\SkyTruth\cv3\source potential\slick_potential_conf_train.csv"
)
df_full_train["perim_over_length"] = (
    df_full_train["perimeter"] / df_full_train["length"]
)
df_full_train.iloc[[0]]

In [ ]:
len(df_full_train)

In [ ]:
sum(
    (df_full_train["max_source_collated_score"] == -5.0)
    & (df_full_train["is_slick"] == False)
)

## Sampling

In [ ]:
# df = pd.concat([
#     df_full_train[df_full_train['is_slick'] == 1],
#     df_full_train[df_full_train['is_slick'] == 0].sample(frac=0.5, random_state=42)
# ])
df = df_full_train
feature_columns = [
    # "length",
    "area",
    # "perimeter",
    "polsby_popper",
    "fill_factor",
    "aspect_ratio_factor",
    "geometry_count",
    "largest_area",
    "median_area",
    "machine_confidence",
    # "machine_cls"
]

## Train Model

In [ ]:
X = df[feature_columns]
y = df["is_slick"].astype(int)
num_of_trials = 1
best_val_accuracy = 0
for trial in tqdm(range(num_of_trials)):
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, stratify=y, test_size=0.25, random_state=42
    )
    print(f"Trial {trial + 1}/{num_of_trials}")
    # rf = RandomForestClassifier(
    #     n_estimators=500,
    #     min_samples_leaf=5,
    #     class_weight="balanced",
    #     n_jobs=-1,
    #     random_state=42,
    # )
    rf = RandomForestClassifier(
        n_estimators=1,
        max_depth=200,
        min_samples_leaf=10,
        min_samples_split=20,
        max_features=None,
        bootstrap=False,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )

    rf.fit(X_train, y_train)
    thres = 0.5
    y_prob = rf.predict_proba(X_val)[:, 1]
    y_pred = (y_prob >= thres).astype(int)
    y_pred_train = rf.predict_proba(X_train)[:, 1]
    y_pred_train = (y_pred_train >= thres).astype(int)
    df["slick_potential"] = rf.predict_proba(X)[:, 1]
    val_accuracy = round(100 * accuracy_score(y_val, y_pred), 6)

    val_f1 = round(100 * f1_score(y_val, y_pred), 6)
    train_f1 = round(100 * f1_score(y_train, y_pred_train), 6)

    print("Validation Accuracy:", val_accuracy, "%", "Validation F1:", val_f1)
    print(
        "Training Accuracy:",
        round(100 * accuracy_score(y_train, y_pred_train), 6),
        "%",
        "Train F1:",
        train_f1,
    )
    if val_accuracy > best_val_accuracy:
        print("Saving Best model")
        dump(
            rf,
            r"C:\Users\ebeva\SkyTruth\cv3\source potential\models"
            + r"\\"
            + f"random_forest_{int(val_accuracy)}_{int(val_f1)}_with_m_conf.joblib",
        )

In [ ]:
X

In [ ]:
sum(df['length'].value_counts() > 1)

In [ ]:
y_val.value_counts()

In [ ]:
# rf = load(r"C:\Users\ebeva\SkyTruth\cv3\source potential\models\random_forest_84_feats_added.joblib")
# df['slick_potential'] = rf.predict_proba(X)[:, 1]

In [ ]:
y_prob = rf.predict_proba(X_val[feature_columns])[:, 1]
# X_val["perim_over_length"] = X_val["perimeter"] / X_val["length"]
X_val["slick_potential"] = y_prob
X_val["is_slick"] = y_val
X_val = X_val.merge(df[["id"]], left_index=True, right_index=True, how="left")

In [ ]:
y_pred = (y_prob >= thres).astype(int)
# compute and plot using the thresholds and y_prob already defined
thresholds, metrics_dict = metrics_vs_threshold(y_prob, y_val, thres=0.5)
metric = "precision"
best_idx = np.argmax(metrics_dict["precision"])
print(
    f"Best threshold: {thresholds[best_idx]:.3f} -> {metric}: {metrics_dict[metric][best_idx]:.4f}"
)

In [ ]:
cm = confusion_matrix(y_val, y_pred)
cm_train = confusion_matrix(y_train, y_pred_train)
plot_cm(cm, "Validation Accuracy")
# plot_cm(cm_train)

y_pred = (y_prob >= thresholds[best_idx]).astype(int)

cm = confusion_matrix(y_val, y_pred)
plot_cm(cm, "Validation Highest Precision")

In [ ]:
import geopandas as gpd
from shapely import wkt

geo_file = r"C:\Users\ebeva\SkyTruth\cv3\source potential\allHITL-1767888944318.csv"
df2 = pd.read_csv(geo_file)
df2["st_astext"] = df2["st_astext"].apply(wkt.loads)
hitl_gdf = gpd.GeoDataFrame(
    df2,
    geometry="st_astext",
    crs="EPSG:4326",  # adjust if coordinates are not lon/lat
)
hitl_gdf["perim_length_ratio"] = hitl_gdf["perimeter"] / hitl_gdf["length"]
hitl_gdf = hitl_gdf[hitl_gdf["hitl_cls"].isin([1, 6, 7, 8])]
hitl_gdf["is_slick"] = hitl_gdf["hitl_cls"].isin([6, 7, 8])

score_series = hitl_gdf.drop_duplicates(subset="id").set_index("id")[
    "max_source_collated_score"
]
df["max_source_collated_score"] = df["id"].map(score_series)
df["max_source_collated_score"] = df["max_source_collated_score"].fillna(-5)
print("Added column; missing values:", df["max_source_collated_score"].isna().sum())

X_val["max_source_collated_score"] = X_val["id"].map(score_series)
X_val["max_source_collated_score"] = X_val["max_source_collated_score"].fillna(-5)
print("Added column; missing values:", X_val["max_source_collated_score"].isna().sum())

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    rf,
    X_val[feature_columns],
    y_val,
    n_repeats=20,
    random_state=42,
    scoring="f1",  # or "recall", "precision"
)

perm_importance = pd.DataFrame(
    {
        "feature": X_val[feature_columns].columns,
        "importance": result.importances_mean,
        "std": result.importances_std,  # optional but very useful
    }
)

# horizontal bar plot of permutation importances with error bars
df_pi = perm_importance.sort_values("importance", ascending=True)

plt.figure(figsize=(8, 4))
plt.barh(
    df_pi["feature"], df_pi["importance"], xerr=df_pi["std"], color="C0", alpha=0.8
)
for i, (imp, std) in enumerate(zip(df_pi["importance"], df_pi["std"])):
    plt.text(imp + std + 0.002, i, f"{imp:.3f}", va="center")
plt.xlabel("Mean permutation importance (f1)")
plt.title("Permutation Feature Importances")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plot_binary_distribution(
    X_val,
    value_col="max_source_collated_score",
    true_label="Slick (True)",
    false_label="Not Slick (False)",
    sentinel_value=None
)
plot_binary_distribution(
    X_val,
    value_col="aspect_ratio_factor",
    true_label="Slick (True)",
    false_label="Not Slick (False)",
    sentinel_value=None
)
plot_binary_distribution(
    X_val,
    value_col="slick_potential",
    true_label="Slick (True)",
    false_label="Not Slick (False)",
    sentinel_value=None
)
plot_binary_distribution(
    X_val,
    value_col="id",
    true_label="Slick (True)",
    false_label="Not Slick (False)",
    sentinel_value=None
)
plot_binary_distribution(
    X_val,
    value_col="machine_confidence",
    true_label="Slick (True)",
    false_label="Not Slick (False)",
    sentinel_value=None
)

In [ ]:
plot_binary_distribution(
    X_val[X_val['max_source_collated_score'] != -5],
    value_col="slick_potential",
    true_label="HITL Slick (True)",
    false_label="HITL Not Slick (False)",
    sentinel_value=None
)

plot_binary_distribution(
    X_val[(X_val['max_source_collated_score'] == -5) | (X_val['is_slick'])],
    value_col="slick_potential",
    true_label="HITL Slick (True)",
    false_label="Unnattributed Slick (False)",
    sentinel_value=None
)

In [ ]:
import matplotlib.pyplot as plt

top = 0.5
bot = 0.0

fp_df = df[df["is_slick"] == False]
tp_df = df[df["is_slick"] == True]

fp_sel = fp_df[
    (fp_df["slick_potential"] > bot) & (fp_df["slick_potential"] < top)
].reset_index(drop=True)
tp_sel = tp_df[
    (tp_df["slick_potential"] > bot) & (tp_df["slick_potential"] < top)
].reset_index(drop=True)

print(
    f"Selected {len(fp_sel)} False Positives and {len(tp_sel)} True Positives from range {top} to {bot}."
)

for i in range(0, 0):
    f_row = fp_sel.iloc[i]
    t_row = tp_sel.iloc[i]

    f_sid = f_row["id"]
    t_sid = t_row["id"]

    f_gdf = hitl_gdf[hitl_gdf["id"] == f_sid]
    t_gdf = hitl_gdf[hitl_gdf["id"] == t_sid]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    # ---------- False Positive ----------
    f_gdf.plot(ax=axes[0], alpha=0.6)

    f_center = geojson_to_lines(f_gdf.iloc[0]["centerlines"])
    plot_lines(f_center, axes[0], color="red", linewidth=0.75)

    axes[0].set_title("False Positive")
    axes[0].annotate(
        f"SID: {f_sid}\nSP: {f_row['slick_potential']:.3f} \nCOL: {f_row['max_source_collated_score']:.3f}",
        xy=(0.02, 0.98),
        xycoords="axes fraction",
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", fc="white", alpha=0.8),
    )

    # ---------- True Positive ----------
    t_gdf.plot(ax=axes[1], alpha=0.6)

    t_center = geojson_to_lines(t_gdf.iloc[0]["centerlines"])
    t_center = geojson_to_lines(t_gdf.iloc[0]["centerlines"])
    plot_lines(t_center, axes[1], color="red", linewidth=0.75)

    axes[1].set_title("True Positive")
    axes[1].annotate(
        f"SID: {t_sid}\nSP: {t_row['slick_potential']:.3f} \nCOL: {t_row['max_source_collated_score']:.3f}",
        xy=(0.02, 0.98),
        xycoords="axes fraction",
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", fc="white", alpha=0.8),
    )

    for ax in axes:
        ax.set_axis_off()

    plt.tight_layout()
    plt.show()

In [ ]:
# scatter plot of slick_potential vs max_collated_score for both classes
col = (
    "max_collated_score"
    if "max_collated_score" in X_val.columns
    else "max_source_collated_score"
)

if col not in X_val.columns:
    raise KeyError(
        "Neither 'max_collated_score' nor 'max_source_collated_score' found in X_val"
    )

plt.figure(figsize=(8, 6))
mask_pos = X_val["is_slick"] == True
mask_neg = ~mask_pos

plt.scatter(
    X_val.loc[mask_neg, col],
    X_val.loc[mask_neg, "slick_potential"],
    color="red",
    alpha=0.6,
    s=20,
    label=f"Not Slick (n={mask_neg.sum()})",
)
plt.scatter(
    X_val.loc[mask_pos, col],
    X_val.loc[mask_pos, "slick_potential"],
    color="green",
    alpha=0.6,
    s=20,
    label=f"Slick (n={mask_pos.sum()})",
)

plt.xlabel(col)
plt.ylabel("slick_potential")
plt.title("slick_potential vs " + col)
plt.grid(alpha=0.3)
plt.legend()

# optionally mark sentinel -5 if present
if (X_val[col] == -5).any():
    plt.axvline(-5, color="gray", linestyle="--", linewidth=1)
    plt.annotate(
        "sentinel = -5",
        xy=(-5, plt.ylim()[1]),
        xytext=(5, -20),
        textcoords="offset points",
    )

plt.show()